# Lesson 1: Reward maximization with a Gaussian policy

In this notebook, a small neural network parameterizes a diagonal two-dimensional Gaussian policy,

$$q_\theta(\mathbf{x}) = \mathcal{N}(\boldsymbol{\mu}_\theta, \operatorname{diag}(\boldsymbol{\sigma}_\theta^2)).$$

We maximize the expected target reward

$$J(\theta) = \mathbb{E}_{\mathbf{x} \sim q_\theta}[R(\mathbf{x})]$$

with two different gradient estimators:

1. the **reparameterization (pathwise) estimator**, and
2. the **log-derivative (score-function) estimator**.

Your task is to implement the two loss functions marked `TODO`. Everything else is provided.

## Setup

The course environment contains all required packages. Follow `../README.md`, then select the project's `.venv` as the notebook kernel.

In [ ]:
# Dependencies are managed by ../pyproject.toml. See ../README.md for setup.

In [ ]:
import math
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib import animation
import torch
from torch import nn
from torch.distributions import Independent, Normal

torch.manual_seed(7)
plt.rcParams["figure.figsize"] = (7, 4)

## 1. The GMM-40 benchmark

We use the GMM-40 benchmark of Midgley et al. (2023), also used by He et al. (2025). It is an equal-weight mixture of 40 isotropic 2D Gaussians. The benchmark draws its means from $[-40,40]^2$ with PyTorch seed $0$, and every component has variance $1$. Its log-density is our reward:

$$R(\mathbf{x}) = \log p_{\mathrm{target}}(\mathbf{x}).$$

High reward therefore means that a state lies close to one or more target modes. There is no minimization objective in this lesson.

In [ ]:
NUM_MODES = 40
MODE_BOUND = 40.0
TARGET_VARIANCE = 1.0
TARGET_STD = math.sqrt(TARGET_VARIANCE)
# Match the reference FAB benchmark exactly: PyTorch RNG with seed 0.
mode_generator = torch.Generator().manual_seed(0)
MODE_LOCATIONS = -MODE_BOUND + 2.0 * MODE_BOUND * torch.rand(
    (NUM_MODES, 2), generator=mode_generator
)
LOG_MIXTURE_WEIGHT = -math.log(NUM_MODES)


def target_reward(states: torch.Tensor) -> torch.Tensor:
    """Log-density reward of the equal-weight 40-mode GMM."""
    standardized = (states.unsqueeze(-2) - MODE_LOCATIONS) / TARGET_STD

    component_log_prob = (
        -0.5 * standardized.square().sum(dim=-1)
        - 2.0 * math.log(TARGET_STD * math.sqrt(2.0 * math.pi))
        + LOG_MIXTURE_WEIGHT
    )
    return torch.logsumexp(component_log_prob, dim=-1)


PLOT_BOUND = 1.4 * MODE_BOUND
plot_axis = torch.linspace(-PLOT_BOUND, PLOT_BOUND, 320)
grid_x, grid_y = torch.meshgrid(plot_axis, plot_axis, indexing="xy")
grid_points = torch.stack((grid_x, grid_y), dim=-1)
reward_on_grid = target_reward(grid_points)
display_reward = reward_on_grid.clamp(min=-1000.0)
reward_levels = torch.linspace(display_reward.min().item(), display_reward.max().item(), 80)
global_grid_index = reward_on_grid.argmax()
GLOBAL_MAX_LOCATION = grid_points.reshape(-1, 2)[global_grid_index]

target_sample_generator = torch.Generator().manual_seed(1)
target_component_ids = torch.randint(NUM_MODES, (5000,), generator=target_sample_generator)
target_samples = MODE_LOCATIONS[target_component_ids] + TARGET_STD * torch.randn(
    (5000, 2), generator=target_sample_generator
)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.8), sharex=True, sharey=True)
for axis in axes:
    axis.contour(
        grid_x, grid_y, display_reward, levels=reward_levels,
        cmap="viridis", linewidths=0.7, alpha=0.9,
    )
    axis.set(
        xlim=(-PLOT_BOUND, PLOT_BOUND), ylim=(-PLOT_BOUND, PLOT_BOUND),
        xticks=(-40, 0, 40), yticks=(-40, 0, 40),
        xlabel="state x₁", ylabel="state x₂", aspect="equal",
    )
axes[0].scatter(
    target_samples[:, 0], target_samples[:, 1],
    s=3, color="tab:blue", alpha=0.45, edgecolors="none", zorder=3,
)
axes[1].scatter(
    MODE_LOCATIONS[:, 0], MODE_LOCATIONS[:, 1],
    s=14, color="tab:blue", edgecolors="none", zorder=3,
)
axes[0].set_title("GMM-40 target samples")
axes[1].set_title("GMM-40 log-density contours")
plt.tight_layout()
output_directory = Path("L1-CostMin")
if not output_directory.is_dir():
    output_directory = Path(".")
LANDSCAPE_PATH = output_directory / "reward_landscape.png"
fig.savefig(LANDSCAPE_PATH, dpi=160, bbox_inches="tight")
plt.show()

## 2. Gaussian policy

Think of this as a one-state decision problem. The policy network receives a constant observation and produces two means and two log standard deviations. We predict `log_std` because exponentiating it guarantees positive standard deviations. `Independent` reinterprets the two scalar Normals as one 2D event. The network output is not clipped.

A single diagonal Gaussian cannot represent 40 separated modes. Because our objective is expected reward (with no entropy bonus), the learned policy will normally choose one reward maximum and become narrow.

In [ ]:
class GaussianPolicy(nn.Module):
    """Neural network that parameterizes a diagonal 2D Gaussian."""

    def __init__(self, hidden_size: int = 32):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(1, hidden_size),
            nn.Tanh(),
            nn.Linear(hidden_size, hidden_size),
            nn.Tanh(),
        )
        self.mean_head = nn.Linear(hidden_size, 2)
        self.log_std_head = nn.Linear(hidden_size, 2)

        # Begin with a broad cloud spanning much of the 40-mode landscape.
        nn.init.zeros_(self.mean_head.weight)
        nn.init.zeros_(self.mean_head.bias)
        nn.init.zeros_(self.log_std_head.weight)
        nn.init.constant_(self.log_std_head.bias, math.log(30.0))

    def forward(self) -> tuple[torch.Tensor, torch.Tensor]:
        device = next(self.parameters()).device
        constant_observation = torch.ones(1, 1, device=device)
        features = self.backbone(constant_observation)

        mean = self.mean_head(features).squeeze(0)
        log_std = self.log_std_head(features).squeeze(0)
        return mean, log_std

    def distribution(self) -> Independent:
        mean, log_std = self()
        return Independent(Normal(mean, log_std.exp()), 1)


policy = GaussianPolicy()
initial_mean, initial_log_std = policy()
print(f"Initial mean: {initial_mean.tolist()}")
print(f"Initial std:  {initial_log_std.exp().tolist()}")

## 3. Loss A: reparameterization trick

A diagonal Gaussian sample can be written as

$$\boldsymbol{\varepsilon} \sim \mathcal{N}(\mathbf{0},I), \qquad \mathbf{x} = \boldsymbol{\mu}_\theta + \boldsymbol{\sigma}_\theta \odot \boldsymbol{\varepsilon}.$$

The random variable $\varepsilon$ does not depend on $\theta$, so gradients can flow through $x$ into the policy. In PyTorch, `Normal.rsample` performs this differentiable sampling step.

**Task:** obtain the policy distribution, draw `num_samples` reparameterized samples, and return the negative mean target reward. PyTorch minimizes losses, so this negative sign implements reward maximization.

In [ ]:
def reparameterization_loss(
    policy: GaussianPolicy, num_samples: int
) -> torch.Tensor:
    """Negative pathwise estimate of E[R(x)] for gradient descent."""
    # TODO: Create q_theta(x) using policy.distribution().
    # TODO: Draw num_samples differentiable samples with rsample.
    # TODO: Return the negative of their mean target reward.
    raise NotImplementedError("Implement the reparameterization loss")

## 4. Loss B: log-derivative trick

The score-function identity, with a baseline $b$ that does not depend on the current sample, is

$$\nabla_\theta J(\theta)
= \mathbb{E}_{x \sim q_\theta}
\left[(R(x)-b)\,\nabla_\theta \log q_\theta(x)\right].$$

Here the sample and its reward must be treated as constants: the gradient should flow only through `log_prob`. PyTorch's `Normal.sample` produces a non-reparameterized sample.

A surrogate loss with the required gradient is

$$L_{\mathrm{score}} = -\frac{1}{N}\sum_i \operatorname{stopgrad}(R(x_i)-b_i)\log q_\theta(x_i).$$

We use a **leave-one-out baseline**: $b_i$ is the mean reward of all samples except $x_i$. It is independent of $x_i$, so it reduces variance without biasing the gradient.

**Task:** sample with `sample`, compute the leave-one-out baseline, detach the reward differences, compute `log_prob`, and return the negative mean surrogate so gradient descent performs reward ascent.

In [ ]:
def log_derivative_loss(
    policy: GaussianPolicy, num_samples: int
) -> torch.Tensor:
    """Negative score-function surrogate for maximizing E[R(x)]."""
    # TODO: Create q_theta(x) using policy.distribution().
    # TODO: Draw num_samples non-reparameterized samples with sample.
    # TODO: Evaluate rewards and compute a leave-one-out mean baseline:
    #       baseline_i = (sum(rewards) - reward_i) / (num_samples - 1)
    # TODO: Detach advantages = rewards - baseline.
    # TODO: Return -mean(advantage * log_prob).
    raise NotImplementedError("Implement the log-derivative loss")

## 5. Check the implementations

A valid loss must be a finite scalar and must create gradients for the policy parameters. This is a structural check, not a proof that the estimator is mathematically correct.

In [ ]:
def check_loss_function(loss_function) -> None:
    torch.manual_seed(0)
    test_policy = GaussianPolicy()
    loss = loss_function(test_policy, num_samples=128)

    assert loss.ndim == 0, "The loss must be a scalar."
    assert torch.isfinite(loss), "The loss must be finite."

    loss.backward()
    gradients = [parameter.grad for parameter in test_policy.parameters()]
    assert all(gradient is not None for gradient in gradients), (
        "Every policy parameter should receive a gradient."
    )
    assert all(torch.isfinite(gradient).all() for gradient in gradients)
    print(f"{loss_function.__name__}: check passed")


check_loss_function(reparameterization_loss)
check_loss_function(log_derivative_loss)

## 6. Train both policies

The reported `estimated_reward` is a fresh Monte Carlo estimate of the true objective and should increase. For the score-function method, the numerical value of its surrogate loss is not itself an estimate of $J(\theta)$; only its gradient is useful.

In [ ]:
def policy_statistics(policy: GaussianPolicy) -> tuple[torch.Tensor, torch.Tensor]:
    with torch.no_grad():
        mean, log_std = policy()
    return mean.cpu(), log_std.exp().cpu()


def train_policy(
    loss_function,
    *,
    seed: int = 7,
    steps: int = 250,
    batch_size: int = 1024,
    learning_rate: float = 3e-3,
    log_every: int = 10,
):
    """Train a fresh policy and record interpretable diagnostics."""
    torch.manual_seed(seed)
    policy = GaussianPolicy()
    optimizer = torch.optim.Adam(policy.parameters(), lr=learning_rate)

    # Reusing these noise vectors makes individual dots move smoothly in the GIF.
    animation_noise = torch.randn(160, 2)
    history = {
        "step": [], "mean": [], "std": [],
        "estimated_reward": [], "samples": [],
    }

    for step in range(steps + 1):
        if step % log_every == 0 or step == steps:
            with torch.no_grad():
                distribution = policy.distribution()
                evaluation_states = distribution.sample((4096,))
                estimated_reward = target_reward(evaluation_states).mean().item()
                mean, std = policy_statistics(policy)

            history["step"].append(step)
            history["mean"].append(mean)
            history["std"].append(std)
            history["estimated_reward"].append(estimated_reward)
            history["samples"].append(mean + std * animation_noise)

        if step == steps:
            break

        optimizer.zero_grad()
        loss = loss_function(policy, batch_size)
        loss.backward()
        nn.utils.clip_grad_norm_(policy.parameters(), max_norm=10.0)
        optimizer.step()

    return policy, history

In [ ]:
pathwise_policy, pathwise_history = train_policy(reparameterization_loss)
score_policy, score_history = train_policy(log_derivative_loss)

for name, trained_policy in [
    ("Reparameterization", pathwise_policy),
    ("Log derivative", score_policy),
]:
    mean, std = policy_statistics(trained_policy)
    print(
        f"{name:20s} -> mean = {mean.numpy().round(3)}, "
        f"std = {std.numpy().round(3)}"
    )

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

for axis, name, trained_policy, color in [
    (axes[0], "Reparameterization", pathwise_policy, "tab:blue"),
    (axes[1], "Log derivative", score_policy, "tab:orange"),
]:
    axis.contour(grid_x, grid_y, display_reward, levels=reward_levels, cmap="viridis", linewidths=0.7, alpha=0.9)
    with torch.no_grad():
        learned_log_density = trained_policy.distribution().log_prob(grid_points)
    peak = learned_log_density.max().item()
    axis.contour(
        grid_x, grid_y, learned_log_density,
        levels=[peak - 4.5, peak - 2.0, peak - 0.5], colors=color, linewidths=2,
    )
    mean, _ = policy_statistics(trained_policy)
    axis.scatter(*mean, marker="*", s=160, color=color, edgecolor="white")
    axis.set(title=f"{name}: final policy", xlim=(-PLOT_BOUND, PLOT_BOUND), ylim=(-PLOT_BOUND, PLOT_BOUND), xlabel="state x₁", ylabel="state x₂", aspect="equal")

axes[2].plot(
    pathwise_history["step"],
    pathwise_history["estimated_reward"],
    label="reparameterization",
)
axes[2].plot(
    score_history["step"],
    score_history["estimated_reward"],
    label="log derivative",
)
axes[2].set(title="Reward maximization", xlabel="optimization step", ylabel="estimated E[R(x)]")
axes[2].legend()
axes[2].grid(alpha=0.2)

plt.tight_layout()
plt.show()

## Discussion

1. Why does the learned Gaussian select one reward maximum instead of covering all 40 modes?
2. Which estimator gives a smoother learning curve? Why?
3. How do the random arrangement of modes and the initial 2D mean affect which local maximum is selected?
4. Remove the leave-one-out baseline and compare the variance of the learning curves.
5. As another extension, add an entropy bonus and observe how it changes the two learned standard deviations.

## 7. Animate the learned samples

Each dot uses the same fixed noise vector in every frame. Its motion therefore shows how the learned mean and standard deviations transform the policy's sample cloud over training. The loss panel shows $-\mathbb E[R(X)]$ for both estimators; its dashed vertical line and point markers track the current frame. Following the paper, thin viridis lines show target log-density contours and blue points show model samples.

In [ ]:
def save_sample_animation(histories, output_path: Path) -> animation.FuncAnimation:
    fig, axes = plt.subplots(
        1, 3, figsize=(14, 4.8), gridspec_kw={"width_ratios": [1, 1, 0.9]}
    )
    landscape_axes = axes[:2]
    loss_axis = axes[2]
    sample_artists = []
    trail_artists = []

    for axis, (name, history, color) in zip(landscape_axes, histories):
        axis.contour(grid_x, grid_y, display_reward, levels=reward_levels, cmap="viridis", linewidths=0.7, alpha=0.9)
        samples = history["samples"][0]
        sample_artists.append(
            axis.scatter(samples[:, 0], samples[:, 1], s=10, alpha=0.55, color=color, edgecolors="none", zorder=4)
        )
        trail_artists.append(axis.plot([], [], color=color, linewidth=2, zorder=3)[0])
        axis.set(xlim=(-PLOT_BOUND, PLOT_BOUND), ylim=(-PLOT_BOUND, PLOT_BOUND), xticks=(-40, 0, 40), yticks=(-40, 0, 40), xlabel="state x₁", ylabel="state x₂", aspect="equal")
        axis.set_title(name)

    curve_colors = ("tab:blue", "tab:orange")
    loss_values = []
    loss_markers = []
    for (name, history, _), curve_color in zip(histories, curve_colors):
        values = [-reward for reward in history["estimated_reward"]]
        loss_values.append(values)
        loss_axis.plot(history["step"], values, color=curve_color, linewidth=2, label=name)
        loss_markers.append(
            loss_axis.scatter([history["step"][0]], [values[0]], s=34, color=curve_color, zorder=3)
        )
    current_step_line = loss_axis.axvline(
        histories[0][1]["step"][0], color="#1F2937", linestyle="--", linewidth=1.4
    )
    loss_axis.set(
        title="Evaluation loss", xlabel="optimization step",
        ylabel="−E[R(X)]",
        xlim=(histories[0][1]["step"][0], histories[0][1]["step"][-1]),
    )
    loss_axis.grid(alpha=0.2)
    loss_axis.legend(fontsize=8)

    step_label = fig.suptitle("")

    def update(frame):
        for (_, history, _), samples_artist, trail_artist in zip(
            histories, sample_artists, trail_artists
        ):
            samples_artist.set_offsets(history["samples"][frame])
            means = torch.stack(history["mean"][: frame + 1])
            trail_artist.set_data(means[:, 0], means[:, 1])
        current_step = histories[0][1]["step"][frame]
        current_step_line.set_xdata([current_step, current_step])
        for values, marker in zip(loss_values, loss_markers):
            marker.set_offsets([[current_step, values[frame]]])
        step_label.set_text(f"Reward-maximizing policy samples — training step {current_step}")
        return [*sample_artists, *trail_artists, *loss_markers, current_step_line, step_label]

    sample_animation = animation.FuncAnimation(
        fig, update, frames=len(histories[0][1]["step"]), interval=500,
    )
    fig.tight_layout(rect=(0, 0, 1, 0.94))
    sample_animation.save(output_path, writer=animation.PillowWriter(fps=2), dpi=100)
    plt.close(fig)
    return sample_animation


output_directory = Path("L1-CostMin")
if not output_directory.is_dir():
    output_directory = Path(".")
GIF_PATH = output_directory / "reward_maximization_training.gif"
sample_animation = save_sample_animation(
    [
        ("Reparameterization", pathwise_history, "tab:blue"),
        ("Log derivative", score_history, "tab:blue"),
    ],
    GIF_PATH,
)
print(f"Saved animation to {GIF_PATH.resolve()}")

from IPython.display import Image, display
display(Image(filename=str(GIF_PATH)))